# Chapter 11

Add your content here.

In [1]:
import numpy as np

# --- 1. The Environment (The "Computer") ---
class MockFileSystem:
    def __init__(self):
        # A simple folder structure
        self.structure = {
            "root": ["Documents", "Pictures", "Downloads"],
            "Documents": ["Work", "Personal"],
            "Work": ["report.txt", "budget.xls"],
            "Pictures": ["photo.jpg"],
            "Downloads": ["installer.exe"]
        }
        self.current_path = "root"
    
    def get_state(self):
        # Returns what the agent "sees" (contents of current folder)
        return f"Current Path: {self.current_path}. Contents: {self.structure.get(self.current_path, [])}"
    
    def step(self, action):
        # Execute an action (e.g., "open Documents")
        contents = self.structure.get(self.current_path, [])
        if action.startswith("open ") and action[5:] in contents:
            new_folder = action[5:]
            # Update path logic (simplified)
            if self.current_path == "root":
                self.current_path = new_folder
            else:
                self.current_path = new_folder # In full version, would handle nested paths
            return f"Opened {new_folder}.", False # False = not done
        
        elif action == "read report.txt" and "report.txt" in contents:
            return "SUCCESS: Read content of report.", True # True = done
            
        return "Error: Invalid Action", False

# --- 2. The Agent S Components ---

class ExperienceMemory:
    """Simulates Retrieval of past knowledge"""
    def __init__(self):
        self.memory = {
            "Find report": ["Go to Documents", "Go to Work", "Read file"]
        }
    
    def retrieve(self, goal):
        # In a real system, we use Cosine Similarity here.
        # Here, we just do a string lookup.
        if "report" in goal:
            return self.memory["Find report"]
        return []

class AgentS:
    def __init__(self):
        self.memory = ExperienceMemory()
    
    def planner_policy(self, state, goal, retrieved_plan):
        """
        The Planner looks at state and memory to decide the Sub-Goal.
        Input: State (String), Goal (String), Plan (List)
        Output: Sub-goal (String)
        """
        contents = state.split("Contents: ")[1]
        
        # Heuristic Logic (Simulating what a Neural Net would learn)
        if "report.txt" in contents:
            return "Read the file"
        
        # If we have a retrieved plan, follow it
        if retrieved_plan:
            current_step_hint = retrieved_plan.pop(0) # Take next step from memory
            return f"Navigate towards {current_step_hint}"
            
        return "Explore Randomly" # Fallback

    def actor_policy(self, state, sub_goal):
        """
        The Actor translates Sub-Goal into specific Action.
        Input: State, Sub-goal
        Output: Action string
        """
        contents = state.split("Contents: ")[1]
        
        if sub_goal == "Read the file":
            return "read report.txt"
        
        if "Go to Documents" in sub_goal and "Documents" in contents:
            return "open Documents"
        
        if "Go to Work" in sub_goal and "Work" in contents:
            return "open Work"
            
        return "wait"

# --- 3. Execution Loop ---

def run_agent_s():
    env = MockFileSystem()
    agent = AgentS()
    goal = "Find and read report.txt"
    
    print(f"--- STARTING TASK: {goal} ---")
    
    # 1. Retrieval Phase
    retrieved_plan = agent.memory.retrieve(goal)
    print(f"Memory Retrieved: {retrieved_plan}")
    
    done = False
    step_count = 0
    
    while not done and step_count < 5:
        # Get State
        state = env.get_state()
        print(f"\n[Step {step_count}] State: {state}")
        
        # Planner Phase (High Level)
        sub_goal = agent.planner_policy(state, goal, retrieved_plan)
        print(f"  Planner decided sub-goal: '{sub_goal}'")
        
        # Actor Phase (Low Level)
        action = agent.actor_policy(state, sub_goal)
        print(f"  Actor executing action: '{action}'")
        
        # Environment Feedback
        result, done = env.step(action)
        print(f"  Result: {result}")
        
        step_count += 1

if __name__ == "__main__":
    run_agent_s()

--- STARTING TASK: Find and read report.txt ---
Memory Retrieved: ['Go to Documents', 'Go to Work', 'Read file']

[Step 0] State: Current Path: root. Contents: ['Documents', 'Pictures', 'Downloads']
  Planner decided sub-goal: 'Navigate towards Go to Documents'
  Actor executing action: 'open Documents'
  Result: Opened Documents.

[Step 1] State: Current Path: Documents. Contents: ['Work', 'Personal']
  Planner decided sub-goal: 'Navigate towards Go to Work'
  Actor executing action: 'open Work'
  Result: Opened Work.

[Step 2] State: Current Path: Work. Contents: ['report.txt', 'budget.xls']
  Planner decided sub-goal: 'Read the file'
  Actor executing action: 'read report.txt'
  Result: SUCCESS: Read content of report.
